In [22]:
# general dependencies
import pandas as pd
import re
import yaml

# pydantic dependencies
from pydantic import BaseModel, field_validator, PrivateAttr, model_validator, ValidationError, Field, StringConstraints
from typing import Literal
from typing import Annotated

# Loading data
from config import YAML_DATA

### Loading YAML Data

In [23]:
# Valve Data
valve_data = YAML_DATA["valve_symbols"]
valve_data['AA']

{'field': 'valve_callout',
 'type': 'base_mounted_valve',
 'desc': '2 POS SGL, RUBBER SEAL',
 'actuation': '1',
 'seal_type': '0',
 'pilot_type': None,
 'back_pressure_check': None,
 'pilot_valve': None,
 'fitting_size': 0,
 'solenoid_qty': 1,
 'x_option': False}

### Sample Data Output From Main Model - valve parser

In [3]:
# call --> manifold.valves
valve_info = [{'qty': 2, 'symbol': 'AB'}, {'qty': 2, 'symbol': 'AT'}, {'qty': 3, 'symbol': 'AA'}, {'qty': 1, 'symbol': 'X'}, {'qty': 2, 'symbol': 'AE'}, {'qty': 5, 'symbol': 'BB'}, {'qty': 2, 'symbol': 'AA'}]

### Base Mounted Valves Model

In [ ]:
class Base_Mounted_Valves(BaseModel):
    # dropping any fields passed that are not declared in model
    model_config = {"extra": "ignore"}

    # --- Explicit Fields from YAML ---
    field: Literal['valve_callout']
    type: Literal['base_mounted_valve']
    actuation: Literal['1', '2', '3', '4', '5', 'A', 'B', 'C']
    seal_type: Literal['', '0', '1']
    pilot_type: Literal['', 'R']
    back_pressure_check: Literal['', 'H']
    pilot_valve: Literal['', 'B', 'K']
    fitting_size: Literal[0, 1, 2, 3]
    solenoid_qty: Literal[1, 2]

    # --- Fields to be inherited from main model class instance ---
    lt_surge_volt_sup_and_coil_type: Literal['R', 'U', 'S', 'Z', 'T', 'V'] # needs to be remapped to orignal catalog syntax, coil type baked into this need to be pulled out
    manual_override: Literal['', 'D', 'E', 'F'] 
    ab_port_size: str

    # In YAML file any empty string or not used values will be set to ~ which is None type
    # To make concatenation easier, these will be converted to empty strings in the model
    @field_validator("seal_type", "pilot_type", "pilot_valve", "back_pressure_check", mode="before")
    def convert_yaml_none_to_strings(cls, v):
        if v is None:
            return ""
        return str(v)
    

obj = Base_Mounted_Valves(**valve_data['AA'])
obj

ValidationError: 3 validation errors for base_mounted_valves
lt_surge_volt_sup_and_coil_type
  Field required [type=missing, input_value={'field': 'valve_callout'...': 1, 'x_option': False}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/missing
manual_override
  Field required [type=missing, input_value={'field': 'valve_callout'...': 1, 'x_option': False}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/missing
ab_port_size
  Field required [type=missing, input_value={'field': 'valve_callout'...': 1, 'x_option': False}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/missing